# H5 Extended Functionalities

In this tutorial, we will show how to extract useful information from the `h5` files generated by DIRAC as in the previous `DiracDataTutorial`, but this time using the `DIRAC` ASE calculator object and methods.

In [ ]:
from ase.build import molecule
from ase.units import Ha
from ase_dirac import DIRAC
import numpy as np
from typing import Optional, Union
import matplotlib.pyplot as plt

## Run the calculation
We start y setting up and running a DIRAC calculation by creating an ASE Atoms object and attaching a `DIRAC` calculator with the `DIRAC.calculator_from_string()` constructor:

In [ ]:
molecule_label = "H2"
ase_molecule = molecule(molecule_label)

inp_string = """
        **DIRAC
        .TITLE
        Relativistic SCF on H2 
        .WAVE F
        **HAMILTONIAN
        .LVCORR
        **WAVE FUNCTIONS
        .SCF
        *SCF
        .PRINT
        1
        **MOLECULE
        *BASIS
        .DEFAULT
        dyall.3zp
    """

ase_molecule.calc: DIRAC = DIRAC.calculator_from_string(inp_string)
ase_molecule.calc.label = ase_molecule.calc.prefix = "H2"

And run the calculation:

In [ ]:
print(
    f"The Hartree-Fock energy of {molecule_label} is ",
    ase_molecule.get_potential_energy() / Ha,
    " Hartree.",
)

It is possible to check the contents of the `h5` file from the ASE calculator object:

In [ ]:
ase_molecule.calc.h5_contents[0:25]

# `h5` Functionality example: Basis definition
In this tutorial, we will show how to obtain useful information such as the definition of the large and small components of the basis, allowing to treat this data externally for any purpose. 
## Large component
The basis information is divided in basis `1` refering tot the large component and `2` refering to the small component. First we will take a look at the large component basis:

In [ ]:
index_L_component= 1 
L_R_array = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_L_component}/center').reshape(-1,3)
L_exps_array = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_L_component}/exponents')
L_l_array = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_L_component}/orbmom')
L_contraction_array = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_L_component}/contractions')
L_n_ao = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_L_component}/n_ao')

print("L component basis information:")
print(f"Number of AOs: {L_n_ao}")
print("Centers (x,y,z):")
print(L_R_array)
print("Exponents:")
print(L_exps_array)
print("Orbital angular momenta:")
print(L_l_array)
print("Contraction coefficients:")
print(L_contraction_array)

With the momenta, we can see that the primitives are $9s2p$. We can compare this with the results from a `grep`: 

In [ ]:
!grep 'Atoms and basis sets' H2.out -A 16

As expected. In the case of the small component:

In [ ]:
index_s_component= 2
S_n_ao = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_s_component}/n_ao')
S_R_array = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_s_component}/center').reshape(-1,3)
S_exps_array = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_s_component}/exponents')
S_l_array = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_s_component}/orbmom')
S_contraction_array = ase_molecule.calc.extract_h5_value(f'input/aobasis/{index_s_component}/contractions')

print("Small component basis information:")
print(f"Number of AOs: {S_n_ao}")
print("Orbital angular momenta:")
print(S_l_array)

That corresponds to $2s9p2d$ as in the `grep` showed. 

## Matrix elements
Let's take a look now at the one electron matrix elements in the `.h5` file:

In [ ]:
overlap = ase_molecule.calc.extract_h5_value('result/operators/ao_matrices/OVERLAP TFFT')
molfieldt = ase_molecule.calc.extract_h5_value('result/operators/ao_matrices/MOLFIELDTFFT')
betamat = ase_molecule.calc.extract_h5_value('result/operators/ao_matrices/BETAMAT FFFT')

print("OVERLAP matrix shape:", overlap.shape)
print("MOLFIELD matrix shape:", molfieldt.shape)
print("BETAMAT matrix shape:", betamat.shape)

Now, `BETAMAT FFFT`, `MOLFIELDTFFT`, and `OVERLAP TFFT` are one-dimensional arrays. Therefore, this means that we have to manipulate them to return to the typical two-dimensional form. To do this, we define the helper functions:


In [ ]:
def retriangularize_and_conjugate(array, n_basis):
    M = np.zeros((n_basis, n_basis), dtype=array.dtype)
    k = 0
    for j in range(n_basis):
        for i in range(j + 1):
            M[i, j] = array[k]
            M[j, i] = np.conjugate(array[k])
            k += 1
    return M


def plot_packed_heatmap(
    array,
    n,
    LC_size: Optional[Union[int, None]] = None,
    title="Packed matrix heatmap",
    cmap="viridis",
    only_LC: bool = False,
):
    M = retriangularize_and_conjugate(array, n)
    Mplot = M.real if np.iscomplexobj(M) else M

    fig, ax = plt.subplots(figsize=(8, 7))
    if only_LC:
        ax.set_xlim(0, LC_size)
        ax.set_ylim(0, LC_size)

    im = ax.imshow(Mplot, cmap=cmap, origin="upper", aspect="auto")
    fig.colorbar(im, ax=ax, label="Matrix element")

    ax.set_xlabel("AO index")
    ax.set_ylabel("AO index")
    ax.set_title(title)

    if LC_size is not None:
        ax.axvline(x=LC_size - 0.5, color="black", linewidth=1.5)
        ax.axhline(y=LC_size - 0.5, color="black", linewidth=1.5)

    plt.tight_layout()
    plt.show()

    return M

We can check the overlap of the large component

In [ ]:
n = int(L_n_ao[0] + S_n_ao[0])
M = plot_packed_heatmap(
    overlap,
    n,
    LC_size=L_n_ao,
    title=f"Overlap [LC: $2(9s2p)$ | SC: $2(2s9p2d)$]",
    only_LC=True,
)

And the overlap of the whole basis:

In [ ]:
M = plot_packed_heatmap(
    overlap, n, LC_size=L_n_ao, title=f"Overlap [LC: $2(9s2p)$ | SC: $2(2s9p2d)$]"
)

Also the `molfield`:

In [ ]:
M = plot_packed_heatmap(
    molfieldt, n, LC_size=L_n_ao, title=f"Molfield  [LC: $2(9s2p)$ | SC: $2(2s9p2d)$]"
)

In [ ]:
M = plot_packed_heatmap(
    betamat, n, LC_size=L_n_ao, title=f"Betamat  [LC: $2(9s2p)$ | SC: $2(2s9p2d)$]"
)